# Abfragen nach Kunden

## 1. Zeige die unterschiedlichen Bezeichnungen für Positionen der Ansprechpartner in den Kunden-Firmen.

![Ku-0010](Screenshots/Ku-0010.png)

In [ ]:
--
SELECT DISTINCT POSITION
FROM kunde;

## 2. Finde Kunden, deren Ansprechpartner eine Inhaber-, Leitungs- oder Manager-Position hat. (43 Zeilen)

![Ku-0020](Screenshots/Ku-0020.png)

In [ ]:
--
SELECT firma AS "Kunde", vorname + ' ' + nachname AS "Ansprechpartner", Position, Telefon
FROM KUNDE
WHERE Position LIKE '%Inhaber%'
   OR Position LIKE '%Leitungs%'
   OR Position LIKE '%Manager%';

## 3. Zeige Firma, Ort und Land von Kunden, deren Firmenname **cocina** oder **comida** enthält.

![Ku-0030](Screenshots/Ku-0030.png)

In [ ]:
--
SELECT firma as 'Kunde', Ort, Land
FROM Kunde
WHERE firma LIKE '%cocina%' OR firma LIKE '%comida%';

## 4. Zeige die Firmen der Kunden nach Ländern gruppiert an. (91 Zeilen)

Sortiere nach Land und Firmennamen.

![Ku-0040](Screenshots/Ku-0040.png)

In [ ]:
--
SELECT Land, Firma
FROM Kunde
ORDER BY Land ASC, Firma ASC;

## 5. Zähle die Anzahl Kunden je Land (21 Zeilen)

Sortiere nach Land.

![Ku-0050](Screenshots/Ku-0050.png)

In [ ]:
--
SELECT Land, COUNT(Land) AS "Anzahl Kunden"
FROM Kunde
GROUP BY Land;

## 6. Ranke die Länder nach Anzahl Kunden absteigend.

In der Nummerierung der Ränge sollen keine Lücken auftreten.

![Ku-0060](Screenshots/Ku-0060.png)

In [ ]:
--
SELECT
    DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS 'Rang',
    Land,
    COUNT(*) AS 'Anzahl Kunden'
FROM Kunde
GROUP BY Land
ORDER BY COUNT(*) DESC;

## 7. Finde Kunden, die 2013 keine Bestellungen gemacht haben.

Zeige mehrere Lösungswege!

![Ku-0070](Screenshots/Ku-0070.png)

In [ ]:
-- unabhängige Unterabfrage
SELECT k.kunde_id, k.firma
FROM Kunde AS k
WHERE k.kunde_id NOT IN (
    SELECT b.kunde_id
    FROM Bestellung AS b
    WHERE YEAR(b.bestelldatum) = 2013
);

In [ ]:
-- korrelierte Unterabfrage
SELECT k.kunde_id, k.firma
FROM Kunde AS k
WHERE NOT EXISTS (
    SELECT 1
    FROM Bestellung AS b
    WHERE b.kunde_id = k.kunde_id
      AND YEAR(b.bestelldatum) = 2013
);

In [87]:
-- JOIN
SELECT k.kunde_id, k.firma
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
WHERE b.kunde_id IS NULL;

(4 rows affected)

kunde_id | firma               
---------+---------------------
86       | Die Wandernde Kuh   
89       | White Clover Markets
90       | Wilman Kala         
91       | Wolski  Zajazd      
(4 rows)

Query execution was canceled.

Total execution time: 00:00:00.047

## 8. Zeige die Firmennamen **aller** Kunden mit der Anzahl ihrer Bestellungen in 2013 an.

Sortiere nach Anzahl absteigend. Finde alternative Lösungen. **(91 Zeilen)**

![Ku-0080](.\Screenshots\Ku-0080.png)

In [88]:
-- mit JOIN
SELECT k.firma, COUNT(b.bestellung_id) AS Anzahl_Bestellungen_2013
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
GROUP BY k.firma
ORDER BY Anzahl_Bestellungen_2013 DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(91 rows affected)

firma                                | Anzahl_Bestellungen_2013
-------------------------------------+-------------------------
Let's Stop N Shop                    | 10                      
LINO-Delicateses                     | 8                       
Magazzini Alimentari Riuniti         | 8                       
Mère Paillarde                       | 8                       
North/South                          | 7                       
Old World Delicatessen               | 7                       
Paris spécialités                    | 7                       
Piccolo und mehr                     | 7                       
Lazy K Kountry Store                 | 7                       
Königlich Essen                      | 7                       
La maison d'Asie                     | 7                       
Drachenblut Delikatessen             | 7                     

In [ ]:
-- mit korrelierter Unterabfrage


## 9. Zeige für 2013 Kunden, Anzahl Bestellungen und Umsatz an.

Sortiere nach Firmenname. Formatiere den Umsatz als Currency. **(91 Zeilen)**

![Ku-0090](Screenshots/Ku-0090.png)

In [96]:
--
SELECT
    k.kunde_id AS [ID],
    k.firma,
    COUNT(DISTINCT b.bestellung_id) AS [Anzahl Bestellungen 2013],
    FORMAT(Coalesce(SUM(bd.verkaufspreis * bd.anzahl),0), 'C', 'de-DE') AS [Umsatz 2013]
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.kunde_id, k.firma
ORDER BY k.firma;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(91 rows affected)

ID | firma                                | Anzahl Bestellungen 2013 | Umsatz 2013
---+--------------------------------------+--------------------------+------------
1  | Alfreds Futterkiste                  | 4                        | 4.873,35 € 
2  | Ana Trujillo Emparedados y helados   | 2                        | 2.261,00 € 
3  | Antonio Moreno Taquería              | 4                        | 6.242,98 € 
4  | Around the Horn                      | 3                        | 1.609,90 € 
5  | Berglunds snabbköp                   | 5                        | 3.624,50 € 
6  | Blauer See Delikatessen              | 2                        | 6.316,00 € 
7  | Blondel père et fils                 | 6                        | 14.756,30 €
8  | Bólido Comidas preparadas            | 3                        | 8.411,40 € 
9  | Bon app'                             | 4                  

## 10. Filtere die vorangehende Abfrage nach Kunden mit über 15 Tsd. Umsatz in 2013.

![Ku-0100](Screenshots/Ku-0100.png)

In [ ]:
--
SELECT
    k.kunde_id AS [ID],
    k.firma,
    COUNT(DISTINCT b.bestellung_id) AS [Anzahl Bestellungen 2013],
    FORMAT(Coalesce(SUM(bd.verkaufspreis * bd.anzahl),0), 'C', 'de-DE') AS [Umsatz 2013]
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.kunde_id, k.firma
HAVING SUM(bd.verkaufspreis * bd.anzahl) > 15000
ORDER BY k.firma

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(6 rows affected)

ID | firma                        | Anzahl Bestellungen 2013 | Umsatz_2013
---+------------------------------+--------------------------+------------
13 | Centro comercial Moctezuma   | 6                        | 15.322,75 €
21 | Familia Arquibaldo           | 6                        | 20.100,20 €
33 | GROSELLA-Restaurante         | 7                        | 25.661,20 €
37 | Hungry Owl All-Night Grocers | 6                        | 28.294,75 €
45 | Let's Stop N Shop            | 10                       | 21.182,68 €
59 | Piccolo und mehr             | 7                        | 19.769,50 €
(6 rows)

Total execution time: 00:00:00.056

## 11. Bewerte Kunden nach Ihrem Umsatz in 2013 wie folgt:

- ab 20000 : "Top"
- ab 15000 : "Super"
- ab 10000 : "Gut"
- ab 5000  : "OK"
- sonst : "Meh!"

**(91 Zeilen)**

![Ku-0101](.\Screenshots\Ku-0101.png)

In [17]:
--
SELECT
  k.kunde_id AS [ID],
  k.firma AS [Kunde],
  FORMAT(COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0), 'C', 'de-DE') AS [Umsatz 2013],
  CASE
    WHEN COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) >= 20000 THEN 'TOP'
    WHEN COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) >= 15000 THEN 'Super'
    WHEN COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) >= 10000 THEN 'Gut'
    WHEN COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) >= 5000 THEN 'OK'
    ELSE 'Meh!'
  END AS [Kundenbewertung]
FROM kunde AS k
LEFT JOIN bestellung AS b ON k.kunde_id = b.kunde_id AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd ON b.bestellung_id = bd.bestellung_id
GROUP BY k.kunde_id, k.firma

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(91 rows affected)

ID | Kunde                                | Umsatz 2013 | Kundenbewertung
---+--------------------------------------+-------------+----------------
23 | Folies gourmandes                    | 7.870,00 €  | OK             
46 | LILA-Supermercado                    | 11.293,02 € | Gut            
69 | Romero y tomillo                     | 2.125,00 €  | Meh!           
29 | Galería del gastrónomo               | 7.350,25 €  | OK             
75 | Split Rail Beer & Ale                | 1.640,90 €  | Meh!           
9  | Bon app'                             | 5.962,15 €  | OK             
15 | Comércio Mineiro                     | 10.289,55 € | Gut            
89 | White Clover Markets                 | 0,00 €      | Meh!           
3  | Antonio Moreno Taquería              | 6.242,98 €  | OK             
52 | Morgenstern Gesundkost               | 10.230,90 € | Gut            
72 | 

## 12. Ranke die Herkunftsländer der Kunden nach ihrem Umsatz in 2013 (21 Zeilen)

![Ku-0110](Screenshots/Ku-0110.png)

In [99]:
--
SELECT
    RANK() OVER (ORDER BY SUM(bd.verkaufspreis * bd.anzahl) DESC) AS Rang,
    k.land,
    FORMAT(Coalesce(SUM(bd.verkaufspreis * bd.anzahl),0), 'C', 'de-DE') AS [Umsatz 2013]
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.land
ORDER BY Rang

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(21 rows affected)

Rang | land           | Umsatz 2013
-----+----------------+------------
1    | USA            | 93.702,56 €
2    | Brasilien      | 89.123,49 €
3    | Frankreich     | 69.712,46 €
4    | Deutschland    | 65.726,67 €
5    | Venezuela      | 62.741,92 €
6    | Großbritannien | 47.493,73 €
7    | Mexiko         | 36.527,13 €
8    | Irland         | 28.294,75 €
9    | Spanien        | 27.810,70 €
10   | Österreich     | 26.531,60 €
11   | Italien        | 20.912,06 €
12   | Argentinien    | 17.882,65 €
13   | Belgien        | 17.122,05 €
14   | Kanada         | 16.188,25 €
15   | Portugal       | 12.074,04 €
16   | Dänemark       | 10.430,15 €
17   | Schweden       | 6.775,74 € 
18   | Schweiz        | 5.976,55 € 
19   | Norwegen       | 3.044,50 € 
20   | Finnland       | 317,75 €   
21   | Polen          | 0,00 €     
(21 rows)

Total execution time: 00:00:00.028

## 13. Pivotiere die Gesamtumsätze nach Ländern und Jahren (21 Zeilen)

![Ku-0120](Screenshots/Ku-0120.png)

In [ ]:
--


## 14. Zeige für jedes Land die beiden Kunden mit dem höchsten Umsatz 2013 an. (39 oder 42 Zeilen)

![Ku-0130](Screenshots/Ku-0130.png)

In [23]:
-- Lösung mit Tabellenausdruck in FROM
SELECT
    land,
    Kunde,
    FORMAT(Umsatz, 'C', 'de-DE') AS [Umsatz 2013]
FROM (
    SELECT
        k.land,
        k.firma AS Kunde,
        COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) AS Umsatz,
        ROW_NUMBER() OVER (PARTITION BY k.land ORDER BY COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) DESC) AS Rang
    FROM kunde AS k
    LEFT JOIN bestellung AS b ON k.kunde_id = b.kunde_id AND YEAR(b.bestelldatum) = 2013
    LEFT JOIN bestelldetail AS bd ON b.bestellung_id = bd.bestellung_id
    GROUP BY k.land, k.firma
) AS Ranked
WHERE Rang <= 2
ORDER BY land, Umsatz DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(39 rows affected)

land           | Kunde                          | Umsatz 2013
---------------+--------------------------------+------------
Argentinien    | Rancho grande                  | 8.564,75 € 
Argentinien    | Océano Atlántico Ltda.         | 6.339,40 € 
Belgien        | Suprêmes délices               | 11.251,20 €
Belgien        | Maison Dewey                   | 5.870,85 € 
Brasilien      | Familia Arquibaldo             | 20.100,20 €
Brasilien      | Ricardo Adocicados             | 13.113,74 €
Dänemark       | Simons bistro                  | 9.719,15 € 
Dänemark       | Vaffeljernet                   | 711,00 €   
Deutschland    | Morgenstern Gesundkost         | 10.230,90 €
Deutschland    | Drachenblut Delikatessen       | 9.328,60 € 
Finnland       | Wartian Herkku                 | 317,75 €   
Finnland       | Wilman Kala                    | 0,00 €     
Frankreich     | Blondel 

In [21]:
-- Lösung mit Tabellenausdruck in WITH
WITH Ranked AS (
    SELECT
        k.land,
        k.firma AS Kunde,
        COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) AS Umsatz,
        ROW_NUMBER() OVER (PARTITION BY k.land ORDER BY COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) DESC) AS Rang
    FROM kunde AS k
    LEFT JOIN bestellung AS b ON k.kunde_id = b.kunde_id AND YEAR(b.bestelldatum) = 2013
    LEFT JOIN bestelldetail AS bd ON b.bestellung_id = bd.bestellung_id
    GROUP BY k.land, k.firma
)
SELECT
    land,
    Kunde,
    FORMAT(Umsatz, 'C', 'de-DE') AS [Umsatz 2013]
FROM Ranked
WHERE Rang <= 2
ORDER BY land, Umsatz DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(39 rows affected)

land           | Kunde                          | Umsatz 2013
---------------+--------------------------------+------------
Argentinien    | Rancho grande                  | 8.564,75 € 
Argentinien    | Océano Atlántico Ltda.         | 6.339,40 € 
Belgien        | Suprêmes délices               | 11.251,20 €
Belgien        | Maison Dewey                   | 5.870,85 € 
Brasilien      | Familia Arquibaldo             | 20.100,20 €
Brasilien      | Ricardo Adocicados             | 13.113,74 €
Dänemark       | Simons bistro                  | 9.719,15 € 
Dänemark       | Vaffeljernet                   | 711,00 €   
Deutschland    | Morgenstern Gesundkost         | 10.230,90 €
Deutschland    | Drachenblut Delikatessen       | 9.328,60 € 
Finnland       | Wartian Herkku                 | 317,75 €   
Finnland       | Wilman Kala                    | 0,00 €     
Frankreich     | Blondel 

In [25]:
-- Lösung mit APPLY
SELECT
    L.land,
    Top2.Kunde,
    FORMAT(Top2.Umsatz, 'C', 'de-DE') AS [Umsatz 2013]
FROM (SELECT DISTINCT land FROM kunde) AS L
CROSS APPLY (
    SELECT TOP 2
        k.firma AS Kunde,
        COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) AS Umsatz
    FROM kunde AS k
    LEFT JOIN bestellung AS b ON k.kunde_id = b.kunde_id AND YEAR(b.bestelldatum) = 2013
    LEFT JOIN bestelldetail AS bd ON b.bestellung_id = bd.bestellung_id
    WHERE k.land = L.land
    GROUP BY k.firma
    ORDER BY Umsatz DESC
) AS Top2
ORDER BY L.land, Top2.Umsatz DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(39 rows affected)

land           | Kunde                          | Umsatz 2013
---------------+--------------------------------+------------
Argentinien    | Rancho grande                  | 8.564,75 € 
Argentinien    | Océano Atlántico Ltda.         | 6.339,40 € 
Belgien        | Suprêmes délices               | 11.251,20 €
Belgien        | Maison Dewey                   | 5.870,85 € 
Brasilien      | Familia Arquibaldo             | 20.100,20 €
Brasilien      | Ricardo Adocicados             | 13.113,74 €
Dänemark       | Simons bistro                  | 9.719,15 € 
Dänemark       | Vaffeljernet                   | 711,00 €   
Deutschland    | Morgenstern Gesundkost         | 10.230,90 €
Deutschland    | Drachenblut Delikatessen       | 9.328,60 € 
Finnland       | Wartian Herkku                 | 317,75 €   
Finnland       | Wilman Kala                    | 0,00 €     
Frankreich     | Blondel 